# Random Forest–XGBoost Common-Weight Ensemble

Kaggle-ready prediction-level ensemble for the closed Random Forest and XGBoost models. The notebook selects one convex Random Forest weight from aligned out-of-sample validation predictions, evaluates that selection with repeated trip-group cross-fitting, freezes the weight, and blends the two existing final submissions. It never loads or retrains the base estimators.


## Ensemble Input And Output Paths

Attach the private `taxi-ensemble-inputs` Kaggle Dataset or override `ENSEMBLE_INPUT_DIR`. The dataset must contain both validation-prediction Parquets, both final submissions, and the manually consolidated `ensemble_manifest.json`.

Create that manifest when assembling the Kaggle Dataset by copying each model's exported sidecar under `models`, then recording each final submission filename under `submissions`. The `...` values below are explanatory placeholders and must be replaced:

```json
{
  "schema_version": 1,
  "models": {
    "random_forest": { "...": "contents of validation_predictions_random_forest.manifest.json" },
    "xgboost": { "...": "contents of validation_predictions_xgboost.manifest.json" }
  },
  "submissions": {
    "random_forest": { "file": "submission_random_forest.csv" },
    "xgboost": { "file": "submission_xgboost.csv" }
  }
}
```


In [ ]:
import os
from pathlib import Path

ENSEMBLE_INPUT_DIR = Path(
    os.environ.get(
        "ENSEMBLE_INPUT_DIR",
        "/kaggle/input/datasets/louishogge/taxi-ensemble-inputs/data",
    )
)
OUTPUT_DIR = Path("/kaggle/working")
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
SUBMISSION_DIR = OUTPUT_DIR / "submission"
PLOT_DIR = OUTPUT_DIR / "plots"
for directory in [ARTIFACT_DIR, SUBMISSION_DIR, PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ENSEMBLE_MANIFEST_PATH = ENSEMBLE_INPUT_DIR / "ensemble_manifest.json"
RF_VALIDATION_PATH = ENSEMBLE_INPUT_DIR / "validation_predictions_random_forest.parquet"
XGB_VALIDATION_PATH = ENSEMBLE_INPUT_DIR / "validation_predictions_xgboost.parquet"
RF_SUBMISSION_PATH = ENSEMBLE_INPUT_DIR / "submission_random_forest.csv"
XGB_SUBMISSION_PATH = ENSEMBLE_INPUT_DIR / "submission_xgboost.csv"

BEST_PARAMS_PATH = ARTIFACT_DIR / "best_params_ensemble.json"
CROSS_VALIDATION_PATH = ARTIFACT_DIR / "ensemble_cross_validation.csv"
RESOLVED_MANIFEST_PATH = ARTIFACT_DIR / "ensemble_manifest.json"
DIAGNOSTIC_PLOT_PATH = PLOT_DIR / "validation_ensemble_diagnostics.png"
ENSEMBLE_SUBMISSION_PATH = SUBMISSION_DIR / "submission_ensemble.csv"

print("Ensemble input directory:", ENSEMBLE_INPUT_DIR)
print("Output directory:", OUTPUT_DIR)


## Imports


In [ ]:
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize_scalar
from scipy.stats import t as student_t
from sklearn.model_selection import GroupKFold


## Common-Weight Configuration

The frozen Random Forest weight is the only learned ensemble parameter. Positive gain means Random Forest error minus ensemble error, so larger values are better.


In [ ]:
SEED = 42
MODEL_NAME = "Random Forest–XGBoost Common-Weight Ensemble"
MODEL_SLUG = "ensemble"
TARGET_COLUMNS = ["END_LONG", "END_LAT"]
PREDICTION_COLUMNS = ["PRED_END_LONG", "PRED_END_LAT"]
EXPORT_COLUMNS = [
    "ROW_ID",
    "TRIP_GROUP",
    "END_LONG",
    "END_LAT",
    "PRED_END_LONG",
    "PRED_END_LAT",
]
SUBMISSION_COLUMNS = ["TRIP_ID", "LATITUDE", "LONGITUDE"]

EXPECTED_VALIDATION_ROWS = 334_581
EXPECTED_SUBMISSION_ROWS = 320
REVIEWED_BASE_SCORES_KM = {
    "random_forest": 2.1194417721912764,
    "xgboost": 2.214959443795627,
}
BASE_SCORE_DRIFT_TOLERANCE_KM = 0.001

RF_WEIGHT_GRID_STEP = 0.05
RF_WEIGHT_XATOL = 1e-4
WEIGHT_TIE_TOLERANCE_KM = 1e-12
CV_REPEATS = 5
CV_FOLDS = 5
REQUIRED_POSITIVE_REPEATS = 4
CONFIDENCE_LEVEL = 0.95


## Reproducibility


In [ ]:
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)


seed_everything()


## Metric And Input Validation Helpers


In [ ]:
def require_existing_path(path, description):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path


def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def save_json(payload, path):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(make_json_safe(payload), file, indent=2, sort_keys=True, allow_nan=False)


def make_json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, (np.integer, np.floating)):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, np.ndarray):
        return make_json_safe(value.tolist())
    if isinstance(value, pd.Series):
        return make_json_safe(value.to_dict())
    if isinstance(value, pd.DataFrame):
        return make_json_safe(value.to_dict(orient="records"))
    if isinstance(value, dict):
        return {str(key): make_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    return value


def haversine_distances_km(y_true_long_lat, y_pred_long_lat):
    truth = np.asarray(y_true_long_lat, dtype=float)
    prediction = np.asarray(y_pred_long_lat, dtype=float)
    if truth.shape != prediction.shape or truth.ndim != 2 or truth.shape[1] != 2:
        raise ValueError(
            f"Expected matching [LONG, LAT] matrices, got {truth.shape} and {prediction.shape}."
        )
    pred_long = np.radians(prediction[:, 0])
    pred_lat = np.radians(prediction[:, 1])
    true_long = np.radians(truth[:, 0])
    true_lat = np.radians(truth[:, 1])
    dlong = true_long - pred_long
    dlat = true_lat - pred_lat
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(pred_lat) * np.cos(true_lat) * np.sin(dlong / 2.0) ** 2
    )
    a = np.clip(a, 0.0, 1.0)
    return 2.0 * 6371.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))


def mean_haversine_km(y_true_long_lat, y_pred_long_lat):
    return float(haversine_distances_km(y_true_long_lat, y_pred_long_lat).mean())


def validate_prediction_frame_structure(frame, expected_rows=None):
    if list(frame.columns) != EXPORT_COLUMNS:
        raise ValueError(
            f"Unexpected validation export columns: {list(frame.columns)!r}; "
            f"expected {EXPORT_COLUMNS!r}."
        )
    if expected_rows is not None and len(frame) != expected_rows:
        raise ValueError(f"Expected {expected_rows} validation rows, got {len(frame)}.")
    if frame["ROW_ID"].duplicated().any():
        raise ValueError("Validation export contains duplicate ROW_ID values.")
    if frame.isna().any().any():
        raise ValueError("Validation export contains missing values.")
    numeric = frame[TARGET_COLUMNS + PREDICTION_COLUMNS].to_numpy(dtype=float)
    if not np.isfinite(numeric).all():
        raise ValueError("Validation export contains non-finite coordinates.")
    expected_groups = frame["ROW_ID"].astype(str).str.rsplit("__prefix_", n=1).str[0]
    if not expected_groups.equals(frame["TRIP_GROUP"].astype(str)):
        raise ValueError("TRIP_GROUP does not match the ROW_ID prefix convention.")


def validate_prediction_export(frame, manifest_entry, file_path):
    validate_prediction_frame_structure(frame, expected_rows=EXPECTED_VALIDATION_ROWS)
    if manifest_entry.get("schema_version") != 1:
        raise ValueError("Unsupported validation prediction manifest schema.")
    if manifest_entry.get("prediction_file") != file_path.name:
        raise ValueError("Prediction file name does not match its manifest entry.")
    if manifest_entry.get("row_count") != len(frame):
        raise ValueError("Prediction row count does not match its manifest entry.")
    if manifest_entry.get("columns") != EXPORT_COLUMNS:
        raise ValueError("Prediction columns do not match the manifest entry.")
    if manifest_entry.get("target_order") != TARGET_COLUMNS:
        raise ValueError("Prediction target order does not match [END_LONG, END_LAT].")


def validate_submission(submission, name):
    if list(submission.columns) != SUBMISSION_COLUMNS:
        raise ValueError(
            f"{name} submission columns are {list(submission.columns)!r}; "
            f"expected {SUBMISSION_COLUMNS!r}."
        )
    if len(submission) != EXPECTED_SUBMISSION_ROWS:
        raise ValueError(
            f"{name} submission has {len(submission)} rows; expected {EXPECTED_SUBMISSION_ROWS}."
        )
    if submission["TRIP_ID"].duplicated().any():
        raise ValueError(f"{name} submission contains duplicate TRIP_ID values.")
    coordinates = submission[["LATITUDE", "LONGITUDE"]].to_numpy(dtype=float)
    if not np.isfinite(coordinates).all():
        raise ValueError(f"{name} submission contains non-finite coordinates.")


## Convex Weight And Uncertainty Helpers


In [ ]:
def blend_predictions(rf_predictions, xgb_predictions, rf_weight):
    rf_weight = float(rf_weight)
    if not 0.0 <= rf_weight <= 1.0:
        raise ValueError(f"RF weight must lie in [0, 1], got {rf_weight}.")
    rf_values = np.asarray(rf_predictions, dtype=float)
    xgb_values = np.asarray(xgb_predictions, dtype=float)
    if rf_values.shape != xgb_values.shape:
        raise ValueError("RF and XGBoost prediction matrices have different shapes.")
    return rf_weight * rf_values + (1.0 - rf_weight) * xgb_values


def select_weight_with_rf_tie_break(weight_score_pairs):
    best_score = min(score for _, score in weight_score_pairs)
    eligible = [
        (weight, score)
        for weight, score in weight_score_pairs
        if score <= best_score + WEIGHT_TIE_TOLERANCE_KM
    ]
    return max(eligible, key=lambda item: item[0])


def optimize_common_weight(truth, rf_predictions, xgb_predictions):
    coarse_weights = np.arange(
        0.0,
        1.0 + RF_WEIGHT_GRID_STEP / 2.0,
        RF_WEIGHT_GRID_STEP,
    )
    coarse_pairs = [
        (
            float(weight),
            mean_haversine_km(
                truth,
                blend_predictions(rf_predictions, xgb_predictions, weight),
            ),
        )
        for weight in coarse_weights
    ]
    coarse_best_weight, _ = select_weight_with_rf_tie_break(coarse_pairs)
    lower = max(0.0, coarse_best_weight - RF_WEIGHT_GRID_STEP)
    upper = min(1.0, coarse_best_weight + RF_WEIGHT_GRID_STEP)

    candidate_pairs = list(coarse_pairs)
    if upper > lower:
        refined = minimize_scalar(
            lambda weight: mean_haversine_km(
                truth,
                blend_predictions(rf_predictions, xgb_predictions, weight),
            ),
            bounds=(lower, upper),
            method="bounded",
            options={"xatol": RF_WEIGHT_XATOL},
        )
        candidate_pairs.append((float(refined.x), float(refined.fun)))
    candidate_pairs.extend([
        (0.0, mean_haversine_km(truth, xgb_predictions)),
        (1.0, mean_haversine_km(truth, rf_predictions)),
    ])
    best_weight, best_score = select_weight_with_rf_tie_break(candidate_pairs)
    return {
        "rf_weight": float(np.clip(best_weight, 0.0, 1.0)),
        "xgboost_weight": float(np.clip(1.0 - best_weight, 0.0, 1.0)),
        "mean_haversine_km": float(best_score),
        "coarse_weights": [weight for weight, _ in coarse_pairs],
        "coarse_scores_km": [score for _, score in coarse_pairs],
    }


def trip_cluster_robust_interval(row_gains_km, trip_groups, confidence=CONFIDENCE_LEVEL):
    values = pd.DataFrame({
        "gain_km": np.asarray(row_gains_km, dtype=float),
        "trip_group": np.asarray(trip_groups, dtype=str),
    })
    grouped = values.groupby("trip_group", sort=False)["gain_km"].agg(["sum", "count"])
    group_count = len(grouped)
    if group_count < 2:
        raise ValueError("At least two trip groups are required for uncertainty estimation.")
    row_count = int(grouped["count"].sum())
    mean_gain = float(grouped["sum"].sum() / row_count)
    influence = grouped["sum"].to_numpy() - mean_gain * grouped["count"].to_numpy()
    variance = (
        group_count / (group_count - 1.0)
        * float(np.square(influence).sum())
        / float(row_count ** 2)
    )
    standard_error = float(np.sqrt(max(variance, 0.0)))
    alpha = 1.0 - confidence
    critical_value = float(student_t.ppf(1.0 - alpha / 2.0, df=group_count - 1))
    return {
        "mean_gain_km": mean_gain,
        "standard_error_km": standard_error,
        "lower_km": mean_gain - critical_value * standard_error,
        "upper_km": mean_gain + critical_value * standard_error,
        "confidence_level": float(confidence),
        "trip_group_count": int(group_count),
        "row_count": row_count,
    }


## Load And Validate Ensemble Inputs

The two validation files must describe the exact same rows, targets, trip groups, preprocessing split, and target order. The existing final submissions are joined only by `TRIP_ID`.


In [ ]:
for input_path, description in [
    (ENSEMBLE_MANIFEST_PATH, "ensemble manifest"),
    (RF_VALIDATION_PATH, "Random Forest validation predictions"),
    (XGB_VALIDATION_PATH, "XGBoost validation predictions"),
    (RF_SUBMISSION_PATH, "Random Forest submission"),
    (XGB_SUBMISSION_PATH, "XGBoost submission"),
]:
    require_existing_path(input_path, description)

ensemble_manifest = load_json(ENSEMBLE_MANIFEST_PATH)
if ensemble_manifest.get("schema_version") != 1:
    raise ValueError("Unsupported ensemble manifest schema version.")
model_manifests = ensemble_manifest.get("models", {})
if set(model_manifests) != {"random_forest", "xgboost"}:
    raise ValueError("Ensemble manifest must contain exactly Random Forest and XGBoost.")
for model_slug in ["random_forest", "xgboost"]:
    if model_manifests[model_slug].get("model_slug") != model_slug:
        raise ValueError(f"Manifest model slug mismatch for {model_slug}.")

rf_frame = pd.read_parquet(RF_VALIDATION_PATH)
xgb_frame = pd.read_parquet(XGB_VALIDATION_PATH)
validate_prediction_export(rf_frame, model_manifests["random_forest"], RF_VALIDATION_PATH)
validate_prediction_export(xgb_frame, model_manifests["xgboost"], XGB_VALIDATION_PATH)

if not rf_frame["ROW_ID"].equals(xgb_frame["ROW_ID"]):
    raise ValueError("RF and XGBoost validation ROW_ID order differs.")
if not rf_frame["TRIP_GROUP"].equals(xgb_frame["TRIP_GROUP"]):
    raise ValueError("RF and XGBoost trip groups differ.")
if not np.array_equal(
    rf_frame[TARGET_COLUMNS].to_numpy(),
    xgb_frame[TARGET_COLUMNS].to_numpy(),
):
    raise ValueError("RF and XGBoost validation targets differ.")

rf_manifest = model_manifests["random_forest"]
xgb_manifest = model_manifests["xgboost"]
if rf_manifest["preprocessed_dataset"] != xgb_manifest["preprocessed_dataset"]:
    raise ValueError("RF and XGBoost preprocessing metadata differs.")

truth = rf_frame[TARGET_COLUMNS].to_numpy(dtype=float)
rf_predictions = rf_frame[PREDICTION_COLUMNS].to_numpy(dtype=float)
xgb_predictions = xgb_frame[PREDICTION_COLUMNS].to_numpy(dtype=float)
trip_groups = rf_frame["TRIP_GROUP"].astype(str).to_numpy()

base_validation_scores = {
    "random_forest": mean_haversine_km(truth, rf_predictions),
    "xgboost": mean_haversine_km(truth, xgb_predictions),
}
for model_slug, score_km in base_validation_scores.items():
    manifest_score = float(model_manifests[model_slug]["validation_metrics"]["mean_haversine_km"])
    if abs(score_km - manifest_score) > 1e-9:
        raise ValueError(f"{model_slug} score does not reproduce its export manifest.")
    reviewed_score = REVIEWED_BASE_SCORES_KM[model_slug]
    if abs(score_km - reviewed_score) > BASE_SCORE_DRIFT_TOLERANCE_KM:
        raise ValueError(
            f"{model_slug} validation score drifted by {score_km - reviewed_score:.6f} km."
        )

submission_manifests = ensemble_manifest.get("submissions", {})
if set(submission_manifests) != {"random_forest", "xgboost"}:
    raise ValueError("Submission manifest must contain exactly Random Forest and XGBoost.")
rf_submission = pd.read_csv(RF_SUBMISSION_PATH, dtype={"TRIP_ID": str})
xgb_submission = pd.read_csv(XGB_SUBMISSION_PATH, dtype={"TRIP_ID": str})
validate_submission(rf_submission, "Random Forest")
validate_submission(xgb_submission, "XGBoost")
for model_slug, file_path in [
    ("random_forest", RF_SUBMISSION_PATH),
    ("xgboost", XGB_SUBMISSION_PATH),
]:
    entry = submission_manifests.get(model_slug, {})
    if entry.get("file") != file_path.name:
        raise ValueError(f"{model_slug} submission file does not match the manifest.")
if set(rf_submission["TRIP_ID"]) != set(xgb_submission["TRIP_ID"]):
    raise ValueError("RF and XGBoost submissions contain different TRIP_ID sets.")

display(pd.DataFrame([
    {"model": "Random Forest", "validation_mean_km": base_validation_scores["random_forest"]},
    {"model": "XGBoost", "validation_mean_km": base_validation_scores["xgboost"]},
]))


## Repeated Trip-Group Cross-Fitting

For each of five shuffled group-fold repetitions, learn the common weight on four folds and apply it only to the untouched fold. Each validation row must be scored exactly once per repetition.


In [ ]:
rf_row_errors_km = haversine_distances_km(truth, rf_predictions)
fold_records = []
repeat_records = []
row_gains_by_repeat = np.empty((CV_REPEATS, len(truth)), dtype=float)

for repeat_index in range(CV_REPEATS):
    splitter = GroupKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=SEED + repeat_index,
    )
    cross_fitted_predictions = np.empty_like(rf_predictions, dtype=float)
    row_coverage = np.zeros(len(truth), dtype=int)

    for fold_index, (train_positions, holdout_positions) in enumerate(
        splitter.split(truth, groups=trip_groups),
        start=1,
    ):
        train_groups = set(trip_groups[train_positions])
        holdout_groups = set(trip_groups[holdout_positions])
        if not train_groups.isdisjoint(holdout_groups):
            raise AssertionError("Trip groups overlap inside a meta-validation fold.")

        fold_optimization = optimize_common_weight(
            truth[train_positions],
            rf_predictions[train_positions],
            xgb_predictions[train_positions],
        )
        fold_weight = fold_optimization["rf_weight"]
        holdout_blend = blend_predictions(
            rf_predictions[holdout_positions],
            xgb_predictions[holdout_positions],
            fold_weight,
        )
        cross_fitted_predictions[holdout_positions] = holdout_blend
        row_coverage[holdout_positions] += 1

        holdout_rf_score = mean_haversine_km(
            truth[holdout_positions],
            rf_predictions[holdout_positions],
        )
        holdout_blend_score = mean_haversine_km(
            truth[holdout_positions],
            holdout_blend,
        )
        fold_records.append({
            "repeat": repeat_index + 1,
            "fold": fold_index,
            "train_rows": int(len(train_positions)),
            "holdout_rows": int(len(holdout_positions)),
            "train_trip_groups": int(len(train_groups)),
            "holdout_trip_groups": int(len(holdout_groups)),
            "rf_weight": float(fold_weight),
            "xgboost_weight": float(1.0 - fold_weight),
            "rf_mean_km": float(holdout_rf_score),
            "blend_mean_km": float(holdout_blend_score),
            "gain_vs_rf_km": float(holdout_rf_score - holdout_blend_score),
        })

    if not np.array_equal(row_coverage, np.ones_like(row_coverage)):
        raise AssertionError("Every validation row must be held out exactly once per repetition.")
    cross_fitted_errors_km = haversine_distances_km(truth, cross_fitted_predictions)
    repeat_gains_km = rf_row_errors_km - cross_fitted_errors_km
    row_gains_by_repeat[repeat_index] = repeat_gains_km
    repeat_records.append({
        "repeat": repeat_index + 1,
        "rf_mean_km": float(rf_row_errors_km.mean()),
        "blend_mean_km": float(cross_fitted_errors_km.mean()),
        "gain_vs_rf_km": float(repeat_gains_km.mean()),
        "positive_gain": bool(repeat_gains_km.mean() > 0.0),
    })

fold_results = pd.DataFrame(fold_records)
repeat_results = pd.DataFrame(repeat_records)
mean_cross_fitted_row_gains_km = row_gains_by_repeat.mean(axis=0)
gain_interval = trip_cluster_robust_interval(
    mean_cross_fitted_row_gains_km,
    trip_groups,
)
positive_repeat_count = int(repeat_results["positive_gain"].sum())
promotion_checks = {
    "positive_repeats": positive_repeat_count,
    "required_positive_repeats": REQUIRED_POSITIVE_REPEATS,
    "positive_repeat_gate_passed": positive_repeat_count >= REQUIRED_POSITIVE_REPEATS,
    "uncertainty_lower_bound_km": float(gain_interval["lower_km"]),
    "uncertainty_gate_passed": bool(gain_interval["lower_km"] > 0.0),
}
ensemble_promoted = bool(
    promotion_checks["positive_repeat_gate_passed"]
    and promotion_checks["uncertainty_gate_passed"]
)
promotion_status = (
    "promoted_ensemble"
    if ensemble_promoted
    else "rejected_keep_random_forest"
)

display(repeat_results)
display(fold_results[["repeat", "fold", "rf_weight", "gain_vs_rf_km"]])
display(pd.Series({**promotion_checks, **gain_interval, "promotion_status": promotion_status}))


## Freeze The Final Common Weight

Cross-fitting supplies the promotion evidence. The deployable scalar is then estimated once from every aligned out-of-sample validation row; its optimized full-validation score remains descriptive rather than an unbiased performance claim.


In [ ]:
full_validation_optimization = optimize_common_weight(
    truth,
    rf_predictions,
    xgb_predictions,
)
frozen_rf_weight = float(full_validation_optimization["rf_weight"])
frozen_xgb_weight = float(1.0 - frozen_rf_weight)
full_validation_gain_km = float(
    base_validation_scores["random_forest"]
    - full_validation_optimization["mean_haversine_km"]
)

decision_summary = pd.Series({
    "promotion_status": promotion_status,
    "recommended_submission": (
        "submission_ensemble.csv"
        if ensemble_promoted
        else "submission_random_forest.csv"
    ),
    "frozen_rf_weight": frozen_rf_weight,
    "frozen_xgboost_weight": frozen_xgb_weight,
    "full_validation_blend_mean_km_descriptive": full_validation_optimization["mean_haversine_km"],
    "full_validation_gain_vs_rf_km_descriptive": full_validation_gain_km,
    "cross_fitted_mean_gain_km": gain_interval["mean_gain_km"],
    "cross_fitted_95pct_lower_km": gain_interval["lower_km"],
    "cross_fitted_95pct_upper_km": gain_interval["upper_km"],
})
display(decision_summary)


## Ensemble Diagnostics


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

axes[0].plot(
    full_validation_optimization["coarse_weights"],
    full_validation_optimization["coarse_scores_km"],
    marker="o",
)
axes[0].axvline(frozen_rf_weight, color="red", linestyle="--")
axes[0].set_xlabel("Random Forest weight")
axes[0].set_ylabel("Mean Haversine distance (km)")
axes[0].set_title("Full-validation weight curve (descriptive)")

axes[1].scatter(
    np.arange(1, len(fold_results) + 1),
    fold_results["rf_weight"],
    alpha=0.75,
)
axes[1].axhline(frozen_rf_weight, color="red", linestyle="--")
axes[1].set_xlabel("Cross-fitted fold")
axes[1].set_ylabel("Selected RF weight")
axes[1].set_ylim(-0.02, 1.02)
axes[1].set_title("Fold-weight stability")

repeat_gain_metres = repeat_results["gain_vs_rf_km"] * 1000.0
bar_colors = np.where(repeat_gain_metres > 0.0, "#2CA02C", "#D62728")
axes[2].bar(repeat_results["repeat"].astype(str), repeat_gain_metres, color=bar_colors)
axes[2].axhline(0.0, color="black", linewidth=1)
axes[2].set_xlabel("Cross-fitting repetition")
axes[2].set_ylabel("Gain versus RF (metres)")
axes[2].set_title("Cross-fitted gain by repetition")

plt.tight_layout()
fig.savefig(DIAGNOSTIC_PLOT_PATH, dpi=160, bbox_inches="tight")
plt.show()


## Blend The Frozen Final Submissions

The same scalar is applied to both coordinates after a validated one-to-one `TRIP_ID` merge. The candidate CSV is always written; the JSON promotion status determines whether it is recommended.


In [ ]:
rf_submission_order = rf_submission["TRIP_ID"].tolist()
merged_submissions = rf_submission.merge(
    xgb_submission,
    on="TRIP_ID",
    how="inner",
    validate="one_to_one",
    suffixes=("_RF", "_XGB"),
    sort=False,
)
if merged_submissions["TRIP_ID"].tolist() != rf_submission_order:
    raise ValueError("The final submission merge did not preserve Random Forest TRIP_ID order.")

ensemble_submission = pd.DataFrame({
    "TRIP_ID": merged_submissions["TRIP_ID"],
    "LATITUDE": (
        frozen_rf_weight * merged_submissions["LATITUDE_RF"]
        + frozen_xgb_weight * merged_submissions["LATITUDE_XGB"]
    ),
    "LONGITUDE": (
        frozen_rf_weight * merged_submissions["LONGITUDE_RF"]
        + frozen_xgb_weight * merged_submissions["LONGITUDE_XGB"]
    ),
})
validate_submission(ensemble_submission, "Ensemble")
ensemble_submission.to_csv(ENSEMBLE_SUBMISSION_PATH, index=False)
display(ensemble_submission.head())


## Save Decision Artifacts


In [ ]:
cross_validation_output = pd.concat([
    fold_results.assign(record_type="fold"),
    repeat_results.assign(record_type="repeat"),
], ignore_index=True, sort=False)
cross_validation_output.to_csv(CROSS_VALIDATION_PATH, index=False)
save_json(ensemble_manifest, RESOLVED_MANIFEST_PATH)

best_params_ensemble = {
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,
    "formula": "prediction = rf_weight * random_forest + (1 - rf_weight) * xgboost",
    "target_order": TARGET_COLUMNS,
    "weights": {
        "random_forest": frozen_rf_weight,
        "xgboost": frozen_xgb_weight,
        "selection_dataset": "all aligned validation rows after cross-fitted gate evaluation",
    },
    "promotion": {
        "status": promotion_status,
        "ensemble_promoted": ensemble_promoted,
        "recommended_submission": (
            "submission_ensemble.csv"
            if ensemble_promoted
            else "submission_random_forest.csv"
        ),
        "checks": promotion_checks,
        "trip_cluster_robust_interval": gain_interval,
    },
    "validation": {
        "row_count": int(len(truth)),
        "trip_group_count": int(pd.Series(trip_groups).nunique()),
        "base_scores_km": base_validation_scores,
        "full_validation_blend_score_km_descriptive": full_validation_optimization["mean_haversine_km"],
        "full_validation_gain_vs_rf_km_descriptive": full_validation_gain_km,
        "cross_validation_repeats": CV_REPEATS,
        "cross_validation_folds": CV_FOLDS,
        "repeat_results": repeat_results,
        "fold_results": fold_results,
    },
    "inputs": {
        "ensemble_manifest_path": str(ENSEMBLE_MANIFEST_PATH),
    },
    "execution": {
        "submission_written": True,
        "submission_path": str(ENSEMBLE_SUBMISSION_PATH),
        "cross_validation_path": str(CROSS_VALIDATION_PATH),
        "diagnostic_plot_path": str(DIAGNOSTIC_PLOT_PATH),
        "local_scoring_completed": False,
        "local_scoring_role": "final audit only; must not alter weight or promotion",
    },
}
save_json(best_params_ensemble, BEST_PARAMS_PATH)

print("Promotion status:", promotion_status)
print("Recommended submission:", best_params_ensemble["promotion"]["recommended_submission"])
print("Ensemble candidate written to:", ENSEMBLE_SUBMISSION_PATH)
print("Best-params artifact written to:", BEST_PARAMS_PATH)


## Decision Interpretation

- `promoted_ensemble` means the candidate beat Random Forest in at least four of five cross-fitting repetitions and the paired 95% trip-cluster interval remained strictly positive.
- `rejected_keep_random_forest` means `submission_ensemble.csv` is retained for audit, but `submission_random_forest.csv` remains the validation-selected submission.
- Local scoring occurs only after the weight and promotion status are frozen and cannot change either decision.
